<a href="https://colab.research.google.com/github/FazeelAr/AI-SE24M-Codes/blob/main/WBC_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics -q

In [ ]:
import zipfile

zip_path = '/content/BCCD.v4-416x416_aug.yolov8.zip'
destination_path = '/content/extracted_data'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(destination_path)

In [ ]:
!cat /content/extracted_data/data.yaml

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 3
names: ['Platelets', 'RBC', 'WBC']

roboflow:
  workspace: joseph-nelson
  project: bccd
  version: 4
  license: MIT
  url: https://universe.roboflow.com/joseph-nelson/bccd/dataset/4

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/extracted_data/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="bccd_finetune"
)

Ultralytics 8.4.107 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/extracted_data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bccd_finetune, nbs=64, nms=F

In [ ]:
metrics = model.val()
print(metrics.box.map)  # mAP50-95

Ultralytics 8.4.107 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 720.0±193.6 MB/s, size: 12.8 KB)
val: Scanning /content/extracted_data/valid/labels.cache... 73 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 73/73 20.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.2it/s 4.1s
                   all         73        967      0.853      0.926      0.932      0.649
             Platelets         42         76      0.816      0.931      0.938      0.518
                   RBC         72        819      0.777      0.847      0.883      0.628
                   WBC         71         72      0.967          1      0.975      0.802
Speed: 6.6ms preprocess, 7.8ms inference, 0.0ms loss, 3.8ms postprocess per image
Results saved to /content/runs/detect/val
0.6492750079

In [ ]:
model = YOLO("runs/detect/bccd_finetune/weights/best.pt")
results = model.predict(source="/content/extracted_data/test/images/BloodImage_00038_jpg.rf.ffa23e4b5b55b523367f332af726eae8.jpg", save=True, conf=0.4)


image 1/1 /content/extracted_data/test/images/BloodImage_00038_jpg.rf.ffa23e4b5b55b523367f332af726eae8.jpg: 640x640 2 Plateletss, 21 RBCs, 1 WBC, 9.3ms
Speed: 3.0ms preprocess, 9.3ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/predict


In [ ]:
from google.colab import files
files.download("runs/detect/bccd_finetune/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>